#COVID-19 in India

**Dataset:** [covid19-in-india @ Kaggle](https://www.kaggle.com/datasets/sudalairajkumar/covid19-in-india)

**Work Plan:**
1. Data loading and cleaning
2. Feature engineering (daily growth, MA7, CFR, recovery rate)
3. EDA: national dynamics, states, waves, heatmap
4. Vaccination analysis
5. SARIMA forecasting with dual-mode validation

## Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
sns.set_style('whitegrid')

PATH_CASES = '/kaggle/input/datasets/sudalairajkumar/covid19-in-india/covid_19_india.csv'
PATH_VACC  = '/kaggle/input/datasets/sudalairajkumar/covid19-in-india/covid_vaccine_statewise.csv'
PATH_TESTS = '/kaggle/input/datasets/sudalairajkumar/covid19-in-india/StatewiseTestingDetails.csv'

## Loading and inspection

In [ ]:
cases = pd.read_csv(PATH_CASES)
vacc  = pd.read_csv(PATH_VACC)
tests = pd.read_csv(PATH_TESTS)

print(f"cases: {cases.shape} | vacc: {vacc.shape} | tests: {tests.shape}")
cases.head(3)

In [ ]:
print("Vacc gaps (%):")
(vacc.isna().mean()*100).round(1).sort_values(ascending=False).head(10)

Problems we found:
46 unique states instead of the expected 36 - typos in Karanataka, Telengana, and artifacts in Bihar****, Maharashtra***
The ConfirmedIndianNational/ForeignNational columns are almost empty
The Confirmed/Cured/Deaths metrics are cumulative

In [ ]:
cases['Date'] = pd.to_datetime(cases['Date'], format='%Y-%m-%d')
cases['State/UnionTerritory'] = cases['State/UnionTerritory'].str.strip()

state_fix = {
    'Bihar****': 'Bihar',
    'Madhya Pradesh***': 'Madhya Pradesh',
    'Maharashtra***': 'Maharashtra',
    'Himanchal Pradesh': 'Himachal Pradesh',
    'Karanataka': 'Karnataka',
    'Telengana': 'Telangana',
    'Daman & Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'Dadra and Nagar Haveli': 'Dadra and Nagar Haveli and Daman and Diu',
}
cases['State'] = cases['State/UnionTerritory'].replace(state_fix)

cases = cases[~cases['State'].isin(['Unassigned', 'Cases being reassigned to states'])].copy()

cases = cases.drop(columns=['Sno', 'Time', 'State/UnionTerritory',
                             'ConfirmedIndianNational', 'ConfirmedForeignNational'])

cases = cases.groupby(['Date', 'State'], as_index=False).agg({
    'Confirmed': 'sum', 'Cured': 'sum', 'Deaths': 'sum'
})

print(f"After cleaning: {len(cases)} term, {cases['State'].nunique()} states/UT")
print(f"Period: {cases['Date'].min().date()} ... {cases['Date'].max().date()}")

## Feature engineering

In [ ]:
cases = cases.sort_values(['State', 'Date']).reset_index(drop=True)

for col in ['Confirmed', 'Cured', 'Deaths']:
    cases[f'Daily_{col}'] = cases.groupby('State')[col].diff().fillna(cases[col])
    cases[f'Daily_{col}'] = cases[f'Daily_{col}'].clip(lower=0)

cases['Active'] = cases['Confirmed'] - cases['Cured'] - cases['Deaths']
cases['Recovery_rate'] = np.where(cases['Confirmed'] >= 100,
                                   cases['Cured']/cases['Confirmed']*100, np.nan)
cases['CFR'] = np.where(cases['Confirmed'] >= 100,
                         cases['Deaths']/cases['Confirmed']*100, np.nan)

cases['Daily_Confirmed_MA7'] = (cases.groupby('State')['Daily_Confirmed']
                                      .transform(lambda s: s.rolling(7, min_periods=1).mean()))
cases['Daily_Deaths_MA7'] = (cases.groupby('State')['Daily_Deaths']
                                   .transform(lambda s: s.rolling(7, min_periods=1).mean()))

cases.head(3)

In [ ]:
national = cases.groupby('Date', as_index=False).agg({
    'Confirmed': 'sum', 'Cured': 'sum', 'Deaths': 'sum',
    'Daily_Confirmed': 'sum', 'Daily_Deaths': 'sum', 'Daily_Cured': 'sum',
    'Active': 'sum'
})
national['Daily_Confirmed_MA7'] = national['Daily_Confirmed'].rolling(7, min_periods=1).mean()
national['Daily_Deaths_MA7']    = national['Daily_Deaths'].rolling(7, min_periods=1).mean()
national['CFR']           = national['Deaths']/national['Confirmed']*100
national['Recovery_rate'] = national['Cured']/national['Confirmed']*100

latest_date = cases['Date'].max()
state_latest = (cases[cases['Date']==latest_date]
                .sort_values('Confirmed', ascending=False)
                .reset_index(drop=True))

print(f"Summary on {latest_date.date()}:")
print(f"  Total cases: {national['Confirmed'].iloc[-1]:,}")
print(f"  Active: {national['Active'].iloc[-1]:,}")
print(f"  Deaths: {national['Deaths'].iloc[-1]:,}")
print(f"  CFR: {national['CFR'].iloc[-1]:.2f}%")

## EDA: National Dynamics

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.bar(national['Date'], national['Daily_Confirmed'], color='#94b4e0', alpha=0.55, label='Raw')
ax.plot(national['Date'], national['Daily_Confirmed_MA7'], color='#1f4e9e', lw=2.3, label='MA7')
ax.set_title('New confirmed cases of COVID-19 in India', fontweight='bold', fontsize=13)
ax.set_ylabel('New cases')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))

peak1 = national.loc[national.loc[national['Date']<'2021-02-01', 'Daily_Confirmed_MA7'].idxmax()]
peak2 = national.loc[national['Daily_Confirmed_MA7'].idxmax()]
ax.annotate(f'Wave 1\npeak: {int(peak1["Daily_Confirmed_MA7"]):,}\n{peak1["Date"].strftime("%d %b %Y")}',
            xy=(peak1['Date'], peak1['Daily_Confirmed_MA7']),
            xytext=(peak1['Date'], peak1['Daily_Confirmed_MA7']+120000),
            fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate(f'Wave 2 (Delta)\npeak: {int(peak2["Daily_Confirmed_MA7"]):,}\n{peak2["Date"].strftime("%d %b %Y")}',
            xy=(peak2['Date'], peak2['Daily_Confirmed_MA7']),
            xytext=(peak2['Date']-pd.Timedelta(days=85), peak2['Daily_Confirmed_MA7']-50000),
            fontsize=9, ha='center', fontweight='bold', color='#8b0000',
            arrowprops=dict(arrowstyle='->', color='#8b0000'))
ax.legend(loc='upper left'); ax.grid(alpha=0.3)


ax = axes[1]
ax.bar(national['Date'], national['Daily_Deaths'], color='#f0a5a5', alpha=0.55, label='Raw')
ax.plot(national['Date'], national['Daily_Deaths_MA7'], color='#8b0000', lw=2.3, label='MA7')
ax.set_title('New COVID-19 deaths in India', fontweight='bold', fontsize=13)
ax.set_ylabel('Deaths per day')
ax.set_xlabel('Date')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.1f}K' if x>=1000 else f'{int(x)}'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(loc='upper left'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Active cases + Recovery rate / CFR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
ax.fill_between(national['Date'], national['Active'], color='#f2b134', alpha=0.7)
ax.plot(national['Date'], national['Active'], color='#c97f00', lw=1.2)
ax.set_title('Active cases (Confirmed − Cured − Deaths)', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1000:.0f}K'))
ax.set_ylabel('Active cases')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(national['Date'], national['Recovery_rate'], color='#2a9d4b', lw=2, label='Recovery rate')
ax.plot(national['Date'], national['CFR'], color='#8b0000', lw=2, label='CFR')
ax.set_title('Recovery rate and CFR (cumulatively)', fontweight='bold')
ax.set_ylabel('%')
ax.set_ylim(0, 105)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Top 15 States by Cases and CFR

In [ ]:
top15 = state_latest.head(15).copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ypos = np.arange(len(top15))
ax.barh(ypos, top15['Confirmed']/1e6, color='#1f4e9e')
ax.set_yticks(ypos); ax.set_yticklabels(top15['State'])
ax.invert_yaxis()
ax.set_xlabel('Confirmed cases, million')
ax.set_title(f'Top 15 states/UTs by cases\n(on {latest_date.strftime("%d %b %Y")})', fontweight='bold')
for i, v in enumerate(top15['Confirmed']/1e6):
    ax.text(v+0.05, i, f'{v:.2f}M', va='center', fontsize=8.5)
ax.grid(alpha=0.3, axis='x')

ax = axes[1]
top15_cfr = top15.sort_values('CFR', ascending=True)
colors = ['#8b0000' if c>2 else '#e07b00' if c>1.5 else '#1f4e9e' for c in top15_cfr['CFR']]
ax.barh(np.arange(len(top15_cfr)), top15_cfr['CFR'], color=colors)
ax.set_yticks(np.arange(len(top15_cfr))); ax.set_yticklabels(top15_cfr['State'])
ax.set_xlabel('CFR, %')
ax.set_title('CFR among the top 15 (red: >2%)', fontweight='bold')
for i, v in enumerate(top15_cfr['CFR']):
    ax.text(v+0.03, i, f'{v:.2f}%', va='center', fontsize=8.5)
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Dynamics of the top 6 states

In [ ]:
top6_states = state_latest.head(6)['State'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
for ax, st in zip(axes.flat, top6_states):
    d = cases[cases['State']==st]
    ax.plot(d['Date'], d['Daily_Confirmed_MA7'], color='#1f4e9e', lw=1.8)
    ax.fill_between(d['Date'], d['Daily_Confirmed_MA7'], alpha=0.25, color='#1f4e9e')
    peak = d.loc[d['Daily_Confirmed_MA7'].idxmax()]
    ax.axvline(peak['Date'], color='#8b0000', ls='--', lw=0.9, alpha=0.7)
    ax.set_title(f"{st}\npeak {int(peak['Daily_Confirmed_MA7']):,} · {peak['Date'].strftime('%d %b %Y')}", fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}K' if x>=1000 else f'{int(x)}'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.grid(alpha=0.3)
fig.suptitle('New Case Trends (MA7): Top 6 States', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Heatmap: Monthly new cases by state

In [ ]:
cases_m = cases.copy()
cases_m['YM'] = cases_m['Date'].dt.to_period('M').dt.to_timestamp()
top20 = state_latest.head(20)['State'].tolist()
pivot = (cases_m[cases_m['State'].isin(top20)]
         .groupby(['State','YM'])['Daily_Confirmed'].sum().unstack(fill_value=0))
pivot = pivot.loc[top20]
pivot_log = np.log10(pivot.replace(0, np.nan))

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot_log, cmap='Blues', cbar_kws={'label':'log10(new cases per month)'},
            linewidths=0.3, linecolor='white',
            xticklabels=[d.strftime('%b %y') for d in pivot.columns], ax=ax)
ax.set_title('Monthly new cases by state (top 20) log scale', fontweight='bold', fontsize=12)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Vaccination Analysis

In [ ]:
vacc = pd.read_csv(PATH_VACC)
vacc.columns = [c.strip() for c in vacc.columns]
vacc['Updated On'] = pd.to_datetime(vacc['Updated On'], format='%d/%m/%Y')
vacc = vacc.rename(columns={
    'Updated On': 'Date',
    'Total Doses Administered': 'Total_Doses',
    'First Dose Administered': 'First_Dose',
    'Second Dose Administered': 'Second_Dose',
    'Male (Doses Administered)': 'Male',
    'Female (Doses Administered)': 'Female',
    'Covaxin (Doses Administered)': 'Covaxin',
    'CoviShield (Doses Administered)': 'CoviShield',
    'Sputnik V (Doses Administered)': 'SputnikV'
})

india_v = vacc[vacc['State']=='India'].sort_values('Date')
india_v = india_v.dropna(subset=['Total_Doses']).reset_index(drop=True)
last = india_v.iloc[-1]

print(f"Vaccination period: {india_v['Date'].min().date()} ... {india_v['Date'].max().date()}")
print(f"Total doses: {last['Total_Doses']/1e6:.1f} million")
print(f"  — first dose: {last['First_Dose']/1e6:.1f} million")
print(f"  — second dose: {last['Second_Dose']/1e6:.1f} million")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(india_v['Date'], india_v['First_Dose']/1e6, color='#1f4e9e', lw=2.2, label='First dose')
ax.plot(india_v['Date'], india_v['Second_Dose']/1e6, color='#2a9d4b', lw=2.2, label='Second dose')
ax.fill_between(india_v['Date'], india_v['First_Dose']/1e6, alpha=0.15, color='#1f4e9e')
ax.fill_between(india_v['Date'], india_v['Second_Dose']/1e6, alpha=0.15, color='#2a9d4b')
ax.set_title('Cumulative vaccination in India', fontweight='bold')
ax.set_ylabel('Doses, million')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
brand_totals = india_v[['Covaxin','CoviShield','SputnikV']].iloc[-1].fillna(0)/1e6
colors_b = ['#e07b00', '#1f4e9e', '#8b0000']
bars = ax.bar(brand_totals.index, brand_totals.values, color=colors_b)
ax.set_title(f'Shares of vaccines on {last["Date"].strftime("%d %b %Y")}', fontweight='bold')
ax.set_ylabel('Doses, million')
total = brand_totals.sum()
for bar, v in zip(bars, brand_totals.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+5, f'{v:.1f}M', ha='center', fontsize=10, fontweight='bold')
    ax.text(bar.get_x()+bar.get_width()/2, v/2, f'{v/total*100:.1f}%', ha='center',
            color='white', fontsize=11, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
vacc_states = vacc[vacc['State']!='India'].copy()
latest_v_date = vacc_states['Date'].max()
vacc_latest = (vacc_states[vacc_states['Date']==latest_v_date]
               .sort_values('Total_Doses', ascending=False).head(15))

fig, ax = plt.subplots(figsize=(11, 6))
ypos = np.arange(len(vacc_latest))
ax.barh(ypos, vacc_latest['First_Dose']/1e6, color='#1f4e9e', label='First dose')
ax.barh(ypos, vacc_latest['Second_Dose']/1e6, color='#2a9d4b', label='Second dose')
ax.set_yticks(ypos); ax.set_yticklabels(vacc_latest['State'])
ax.invert_yaxis()
ax.set_xlabel('Doses, million')
ax.set_title(f'Top 15 states for vaccination (on {latest_v_date.strftime("%d %b %Y")})', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## SARIMA Forecasting

Model: **SARIMA(1,1,1)(1,1,1,7)** with weekly seasonality

Approach: train/test split with a 30-day holdout, comparison with a naive baseline (average over the last 7 days of train)


In [ ]:
ts = national.set_index('Date')['Daily_Confirmed'].astype(float)
ts = ts[ts.index >= '2020-03-01']

adf = adfuller(ts.dropna())
print(f"ADF test: stat={adf[0]:.3f}, p-value={adf[1]:.4f}")

horizon = 30
train, test = ts.iloc[:-horizon], ts.iloc[-horizon:]
print(f"Train: {len(train)} days | Test: {len(test)} days ({test.index.min().date()}..{test.index.max().date()})")

model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,7),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False, maxiter=200)
print(f"AIC = {fit.aic:.1f}")

fc = fit.get_forecast(steps=horizon)
pred_mean = fc.predicted_mean.clip(lower=0)
ci = fc.conf_int(alpha=0.05)
baseline = pd.Series(train.rolling(7).mean().iloc[-1], index=test.index)

def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(((y_true - y_pred)**2).mean())
    mape = mean_absolute_percentage_error(y_true, y_pred)*100
    print(f"  {label:12s}: MAE={mae:>10,.0f} | RMSE={rmse:>10,.0f} | MAPE={mape:>5.1f}%")
    return mae, rmse, mape

print("\n30-day metrics hold-out:")
m_arima = metrics(test.values, pred_mean.values, 'SARIMA')
m_base  = metrics(test.values, baseline.values, 'Naive MA7')

In [ ]:
full_fit = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,7),
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
future = full_fit.get_forecast(steps=30)
future_mean = future.predicted_mean.clip(lower=0)
future_ci   = future.conf_int(alpha=0.05).clip(lower=0)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

ax = axes[0]
recent = ts[ts.index >= '2021-04-01']
ax.plot(recent.index, recent.values, color='#333', lw=1.5, label='Fact')
ax.plot(pred_mean.index, pred_mean.values, color='#1f4e9e', lw=2.3, label='SARIMA')
ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], color='#1f4e9e', alpha=0.2, label='95% CI')
ax.plot(baseline.index, baseline.values, color='#8b0000', lw=1.5, ls='--', label='Naive MA7')
ax.axvline(train.index[-1], color='gray', ls=':', alpha=0.7)
ax.set_title(f'Validation (30-day hold-out) | SARIMA MAE={m_arima[0]:,.0f} vs Naive MAE={m_base[0]:,.0f}',
             fontweight='bold', fontsize=11)
ax.set_ylabel('New cases/day')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
tail = ts[ts.index >= '2021-05-01']
ax.plot(tail.index, tail.values, color='#333', lw=1.5, label='History')
ax.plot(future_mean.index, future_mean.values, color='#2a9d4b', lw=2.3, label='30-day forecast')
ax.fill_between(future_ci.index, future_ci.iloc[:,0], future_ci.iloc[:,1], color='#2a9d4b', alpha=0.2, label='95% CI')
ax.axvline(ts.index[-1], color='gray', ls=':', alpha=0.7)
ax.set_title(f'Forecast for 30 days ahead (с {(ts.index[-1]+pd.Timedelta(days=1)).date()})', fontweight='bold')
ax.set_ylabel('New cases/day')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Residue diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
resid = full_fit.resid
axes[0].plot(resid.index, resid.values, color='#1f4e9e', alpha=0.7)
axes[0].axhline(0, color='k', ls='--', alpha=0.5)
axes[0].set_title('Remains of SARIMA', fontweight='bold')
axes[0].grid(alpha=0.3)

plot_acf(resid.dropna(), lags=30, ax=axes[1])
axes[1].set_title('ACF of residues', fontweight='bold')
plt.tight_layout()
plt.show()

## Comparison of modes: growth phase vs. plateau

In [ ]:
scenarios = {
    'growth_phase':  {'train_end': '2021-04-14', 'horizon': 14, 'title': 'Growth phase of the 2nd wave'},
    'plateau_phase': {'train_end': '2021-07-12', 'horizon': 30, 'title': 'Plateau phase (after the wave)'},
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (sc_name, sc) in zip(axes, scenarios.items()):
    end_date = pd.Timestamp(sc['train_end'])
    h = sc['horizon']
    train_s = ts[ts.index <= end_date]
    test_s  = ts[(ts.index > end_date) & (ts.index <= end_date+pd.Timedelta(days=h))]

    fit_s = SARIMAX(train_s, order=(1,1,1), seasonal_order=(1,1,1,7),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
    fc_s  = fit_s.get_forecast(steps=h)
    pred_s = fc_s.predicted_mean.clip(lower=0)
    ci_s = fc_s.conf_int(alpha=0.05).clip(lower=0)
    naive_s = pd.Series(train_s.rolling(7).mean().iloc[-1], index=test_s.index)

    mae_sa = mean_absolute_error(test_s.values, pred_s.values)
    mae_n  = mean_absolute_error(test_s.values, naive_s.values)
    mape_sa = mean_absolute_percentage_error(test_s.values, pred_s.values)*100
    mape_n  = mean_absolute_percentage_error(test_s.values, naive_s.values)*100

    window = ts[(ts.index >= end_date - pd.Timedelta(days=35)) & (ts.index <= end_date + pd.Timedelta(days=h))]
    ax.plot(window.index, window.values, color='#333', lw=1.5, label='Fact')
    ax.plot(pred_s.index, pred_s.values, color='#1f4e9e', lw=2.3, label=f'SARIMA (MAPE={mape_sa:.1f}%)')
    ax.fill_between(ci_s.index, ci_s.iloc[:,0], ci_s.iloc[:,1], color='#1f4e9e', alpha=0.2)
    ax.plot(naive_s.index, naive_s.values, color='#8b0000', lw=1.6, ls='--', label=f'Naive MA7 (MAPE={mape_n:.1f}%)')
    ax.axvline(end_date, color='gray', ls=':', alpha=0.7)
    winner = 'SARIMA' if mae_sa<mae_n else 'Naive'
    winner_color = '#1f4e9e' if mae_sa<mae_n else '#8b0000'
    ax.set_title(f'{sc["title"]}\n← wins {winner} →', fontweight='bold', color=winner_color)
    ax.set_ylabel('New cases/day')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1000:.0f}K'))
    ax.tick_params(axis='x', rotation=45)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Results

India has experienced two waves of COVID-19, each differing in magnitude by a factor of 4.2. The first wave peaked on September 17, 2020, with 93,198 cases per day, while the second (Delta) wave peaked on May 9, 2021, with 391,279 cases per day. Daily deaths followed the same pattern: about 1.2K/day at the peak of the first wave versus ~4K/day in the second. By August 2021, the country had stabilized at 35-40K cases per day. As of August 11, 2021, there were 32,036,511 cases, 429,179 deaths, 386,351 active cases, and a national CFR of 1.34%. Maharashtra was the state's geographical epicenter: 6.36 million cases (20% of the national total) and a CFR of 2.11%, the highest among the top 15 states, significantly above the national level and indicating an overload of the healthcare system at the peak of the second wave, which peaked in the state at 65,447 cases on April 25, 2021. Kerala, however, exhibits a paradox: 3.59 million cases: second place in absolute numbers, with the lowest CFR in the top 15 (0.50%) and the highest number of active cases at the time of the cutoff (172K). Two likely explanations: better case counting means less hidden fatality; or the wave arrived in the state later, so there are more active cases. State peaks are tightly clustered in late April - late May 2021 (Uttar Pradesh on April 28, Maharashtra on April 25, Karnataka on May 10, Kerala on May 13, Andhra Pradesh on May 21, Tamil Nadu on May 26), confirming the synchronicity of the nationwide surge.
In terms of vaccinations, 513.2 million doses were administered over 7 months (January 16 - August 9, 2021): 400.2 million first doses and 113.1 million second doses. With a population of ~1.4 billion, this means ~29% have received at least one dose and only ~8% are fully vaccinated. Covishield dominates: 446.8 million doses (87.7%), Covaxin accounts for 62.4 million (12.2%), and Sputnik V has a marginal share (0.1%). To forecast daily cases, the SARIMA(1,1,1)(1,1,1,7) model with weekly seasonality was used. The Dickey-Fuller test yielded a p-value of 0.0061, but structural breaks between waves require differentiating d≥1. The main methodological conclusion: SARIMA works only in a suitable data regime. In the growth phase (April 14-27, 2021), the SARIMA MAPE was 21.4% versus 46.2% for the naive baseline (average over the last 7 days); the model wins by about 2 times. In the plateau phase (July 13 - August 11, 2021), the situation is reversed: the SARIMA MAPE was 20.0% versus 10.7% for the naive baseline; now the constant wins by 2 times. This is a classic illustration of the bias-variance trade-off: a complex model pays off only when there is a trend and seasonality in the data; In a stabilized series, it wastes degrees of freedom on noise and underperforms compared to a simple average.
Diagnostics of the residuals revealed limitations of the model: the residuals are heteroscedastic, and the variance increases sharply at the peak of the second wave (March-May 2021), which violates SARIMA assumptions and makes the 95% confidence intervals too low during these periods. The residual ACF shows residual autocorrelation at lags 3 and 6, so the model is not perfect, but acceptable.
The main limitations of the study: there is no state population data, it is impossible to calculate per-capita metrics, and Maharashtra is also the most populous state, so absolute differences with smaller states are overstated. Converting cumulative metrics to daily increments yields negative values ​​on days of retroactive reporting adjustments; zeroing them using clip(lower=0) systematically underestimates the mean. The positivity rate has a 65% missingness rate, meaning conclusions based on this metric are only qualitative. The CFR is not equal to the IFR: confirmed cases are just the tip of the iceberg, especially in 2020 with limited testing. The forecasting is purely univariate, without exogenous regressors (mobility index, stringency index, hospitalizations), the inclusion of which could significantly improve the model. Possible future directions for this work include calculating the R_t (effective reproduction number) using the method of Cori et al., replacing SARIMA with LightGBM with lagged features, and assessing the impact of lockdowns using difference-in-differences between states.
I welcome your comments and feedback on this work.